# Create BCRF Awards (Breast Cancer Research Foundation)

Creates Breast Cancer Research Foundation awards from BCRF's own "Meet Our
Researchers" directory at https://www.bcrf.org/researchers/ (every profile is in
`researchers-sitemap.xml`; method 5, static HTML). One row per BCRF-funded
investigator profile: name, institution, location, the one-line **Research
area** statement of what the BCRF grant funds (used as `display_name`), the
grant narrative (Impact / Progress Thus Far / What's next -> `description`),
and "BCRF Investigator Since" (-> `start_year`). **217 awards shipped**
(314 profiles parsed 2026-10-01; 1 emeritus page skipped).

**No grant number, amount or grant dates on the profiles.** `funder_award_id` is
the synthetic `BCRF-INV-{profile slug}`; it collapses onto 0 existing citation
stubs. `amount`/`currency` are NULL (BCRF does not publish per-investigator
amounts in the directory; §6.7 amount check waived). `start_year` = first year of
BCRF funding, `start_date` = Jan 1 of that year (§2.3 year-only pattern); no end.

**Overlap with BCRF's own Crossref grant deposits (§2.1.1).** BCRF registers
grant DOIs itself (prefix 10.63055, award numbers `BCRF-YY-NNN`, the form citing
works use; 107 deposits, mostly the 2026 cycle). They are already in
`openalex_awards_raw` as `crossref_work` rows (priority 1) with title, abstract,
amount, dates and PI ORCID. The script matches each profile to those deposits
(lead investigator family name + first given initial; all 97 matches checked
against the deposit's institution) and does **not** ship the 97 matched
investigators (`excluded = '1'`, `exclusion_reason = 'covered_by_crossref_grant'`,
deposit award number kept in `crossref_award`), so the directory cannot duplicate
or, at a higher priority, overwrite BCRF's richer Crossref record. This notebook
ships only `excluded = '0'` rows.

**Twins:** none (International Breast Cancer Research Foundation F4320315947 and
BCRF of Alabama F4320316599 are different organisations).

**Prerequisites:** run `scripts/local/bcrf_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/bcrf/bcrf_projects.parquet` (§1.4 shrink guard).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320306260`
- display_name: Breast Cancer Research Foundation
- ror_id / doi: from `openalex.funders.funders` (ror 0348ff195, doi 10.13039/100001006)

**Priority:** `539` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.bcrf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/bcrf/bcrf_projects.parquet`;

In [ ]:
%sql
SELECT excluded, exclusion_reason, COUNT(*) as n
FROM openalex.awards.bcrf_raw GROUP BY 1, 2 ORDER BY 1;

In [ ]:
%sql
DESCRIBE openalex.awards.bcrf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.bcrf_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320306260 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320306260;

## Step 2: Create BCRF Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.bcrf_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320306260  -- Breast Cancer Research Foundation
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    -- the profile's one-line "Research area" statement of what the grant funds
    REGEXP_REPLACE(TRIM(g.research_area), '[.]$', '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CAST(NULL AS STRING) as funder_scheme,
    'bcrf_researchers' as provenance,
    TRY_TO_DATE(CONCAT(g.investigator_since, '-01-01'), 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.investigator_since AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- struct field order MUST match openalex_awards_raw.
    named_struct(
        'given_name', NULLIF(TRIM(g.given_name), ''),
        'family_name', NULLIF(TRIM(g.family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', NULLIF(TRIM(g.country), ''),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.bcrf_raw g
CROSS JOIN src_funder f
WHERE g.excluded = '0'
  AND g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != ''
  AND NULLIF(TRIM(g.family_name), '') IS NOT NULL;

In [ ]:
%sql
-- Shape check (§2.1.1): every funder_award_id is the synthetic BCRF-INV-<slug>; expect 0 rows.
SELECT funder_award_id FROM openalex.awards.bcrf_awards
WHERE NOT funder_award_id RLIKE '^BCRF-INV-[a-z0-9-]+$'
LIMIT 20;

In [ ]:
%sql
-- Must be 0: no shipped investigator is the lead investigator of an existing BCRF
-- crossref_work grant (those are skipped in the script; this guards the match).
SELECT b.funder_award_id, b.lead_investigator.given_name, b.lead_investigator.family_name, c.funder_award_id as crossref_award
FROM openalex.awards.bcrf_awards b
JOIN openalex.awards.openalex_awards_raw c
  ON c.funder_id = 4320306260 AND c.provenance = 'crossref_work'
 AND LOWER(c.lead_investigator.family_name) = LOWER(b.lead_investigator.family_name)
 AND LOWER(LEFT(c.lead_investigator.given_name, 1)) = LOWER(LEFT(b.lead_investigator.given_name, 1));

In [ ]:
%sql
-- Never shrink what may already be published: refuse to replace a larger earlier load.
SELECT assert_true(
    (SELECT COUNT(*) FROM openalex.awards.bcrf_awards) >=
    (SELECT COUNT(*) FROM openalex.awards.openalex_awards_raw WHERE provenance = 'bcrf_researchers' AND priority = 539),
    'bcrf_awards is smaller than the rows already in openalex_awards_raw; refusing to shrink'
) as no_shrink;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'bcrf_researchers' AND priority = 539;

-- Insert into openalex_awards_raw with priority.
-- Priority 539: direct-from-funder ingest of BCRF's researcher directory.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    539 as priority
FROM openalex.awards.bcrf_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_bcrf_awards, COUNT(DISTINCT id) as distinct_ids FROM openalex.awards.bcrf_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, lead_investigator.affiliation.country, landing_page_url
FROM openalex.awards.bcrf_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.bcrf_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail (expect 1 each).
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.bcrf_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.bcrf_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 coverage (expect 100% title/PI, ~99% description/institution, 0% amount by design).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(lead_investigator.affiliation.country) as has_country
FROM openalex.awards.bcrf_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'bcrf_researchers'
GROUP BY provenance, priority;